# ExplainHub — Llama 3 8B documentation enhancer (Soup)

This notebook runs the real fine-tune on a Colab GPU. It assumes the dataset was
built locally with `training/scripts/build-dataset.mjs` and that you upload
`training/data/train.jsonl`.

The training config is `training/soup.yaml`: Llama 3 8B Instruct, 4-bit QLoRA,
LoRA r=16 / alpha=32, ChatML rows rendered with the Llama 3 template, and
answer-only loss (the production prompt is masked).


## 1. Check the GPU and install Soup + Liger

A T4 (16 GB) works with `max_length: 6144` because Liger Kernel's fused linear
cross-entropy never materializes the 11 GB logits tensor. An L4 (24 GB) or A100
(40 GB) is faster and can use 8192 tokens.

The install cell pins `liger-kernel==0.8.5`, which requires `transformers>=5`
(Colab ships 5.x). Do **not** install `cut-cross-entropy`: its PyPI wheel does
not include the `transformers` patch Soup imports, so Soup silently falls back
to the full logits tensor and OOMs.

In [ ]:
!nvidia-smi
!pip install -q "soup-cli[train]" "liger-kernel==0.8.5"
# `soup` runs under /usr/bin/python3, which may see a different
# site-packages than `pip` (usually /usr/local/bin/python). Install Liger
# into the interpreter Soup actually uses and verify from that interpreter.
!/usr/bin/python3 -c "import liger_kernel" 2>/dev/null || /usr/bin/python3 -m pip install -q --break-system-packages "liger-kernel==0.8.5"
!/usr/bin/python3 -c "import liger_kernel, transformers; print('liger', liger_kernel.__version__, '| transformers', transformers.__version__)"
!soup --version
!soup doctor


## 2. Clone the repository

The training project lives on the `training/soup-finetune` branch. Push that
branch before running the notebook, or change `BRANCH` to whichever branch you
pushed. Replace `REPO_URL` if you are working from a fork. The notebook uses the
`soup.yaml` from the cloned repository, including `max_length: 6144` and
`use_liger: true`.


In [ ]:
REPO_URL = "https://github.com/tadeolmd-ubu/ExplainHub.git"
BRANCH = "training/soup-finetune"

import os
if not os.path.isdir("ExplainHub"):
    !git clone -q $REPO_URL
%cd ExplainHub
!git fetch -q origin $BRANCH
!git checkout -q $BRANCH
!git pull -q origin $BRANCH
%cd training
!git branch --show-current
!pwd


## 3. Upload the training file

The generated `train.jsonl` is intentionally git-ignored (it is reproducible
from `training/datasets/pairs/`). Upload it from your machine.


In [ ]:
from google.colab import files

uploaded = files.upload()
name = next(iter(uploaded))
!mkdir -p data
!mv "$name" data/train.jsonl
!wc -l data/train.jsonl


## 4. Validate the rows before spending GPU time

`soup data validate` checks the ChatML shape. `soup data doctor` loads the
tokenizer, checks the chat template and the answer-only mask, and reports
truncation risk at `max_length: 4096`.


In [ ]:
!soup data validate data/train.jsonl --format chatml


In [ ]:
!soup data stats data/train.jsonl


In [ ]:
!soup data doctor data/train.jsonl \
  --model unsloth/llama-3-8b-Instruct \
  --format chatml \
  --max-length 6144


## 5. Train

The config uses Liger Kernel (`training.use_liger: true`) to keep the 8B QLoRA
run inside 16 GB. Soup's pre-flight gate still prices the full logits tensor, so
the run needs `--allow-oom-attempt`; the fused loss is what makes it fit.

Expected log lines:

```text
Liger Kernel enabled: fused RMSNorm, SwiGLU, CrossEntropy, RoPE
```

The training cell also sets `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`,
which cuts allocator fragmentation on a T4.

If the T4 still runs out of memory, set `data.max_length: 4096` in `soup.yaml`,
rebuild the dataset locally with `--max-chars 16000` so the over-long row moves
to `long.jsonl` instead of being truncated, and re-upload `train.jsonl`.

In [ ]:
# expandable_segments reduces CUDA allocator fragmentation on the 16 GB T4.
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True soup train --config soup.yaml --allow-oom-attempt --yes


## 6. Export the GGUF

This produces a quantized file you can download and run with Ollama locally.


In [ ]:
!soup export \
  --model ./output \
  --format gguf \
  --quant q4_k_m \
  --output explainhub-docs.q4_k_m.gguf


In [ ]:
from google.colab import files

files.download("explainhub-docs.q4_k_m.gguf")


## 7. Deploy to Ollama on your machine

Create a `Modelfile` next to the downloaded GGUF:

```text
FROM ./explainhub-docs.q4_k_m.gguf
```

Then register it:

```bash
ollama create explainhub-docs -f Modelfile
ollama run explainhub-docs
```

Alternatively, copy the Soup adapter directory (`training/output/`) back to your
machine and let Soup convert and deploy it:

```bash
cd training
soup export \
  --model ./output \
  --format gguf \
  --quant q4_k_m \
  --deploy ollama \
  --deploy-name explainhub-docs
```


## 8. Point ExplainHub at the model

Set the model name in `.env`:

```env
OLLAMA_MODEL=explainhub-docs
```

Then run a normal Markdown analysis. `AiEnhancer.enhanceMarkdown` will use the
same prompt the model was trained on, because the prompt is imported from
`src/modules/ai-enhancer/prompt/promptMdEnhancer.js` in both places.


## 9. Measure the result

Run the held-out evaluation locally, with both models available in Ollama:

```bash
node training/scripts/evaluate.mjs \
  --base llama3:8b \
  --tuned explainhub-docs \
  --limit 20
```

The script writes `training/data/eval-report.md` and `.json` with structural
and textual metrics plus the per-document outputs.


## Troubleshooting

- **Soup refuses to start: "Predicted peak VRAM … exceeds"**: pass
  `--allow-oom-attempt`. The gate counts the full logits tensor even though
  Liger's fused loss removes it.
- **`Liger Kernel: no matching architecture found`**: Liger is not installed in
  the interpreter Soup runs. Re-run the install cell; it installs into
  `/usr/bin/python3` and verifies the import from that same interpreter.
- **CUDA out of memory on a T4**: confirm the log says
  `Liger Kernel enabled` and `training.use_liger: true` is in `soup.yaml`. If it
  still runs out, lower `data.max_length` to 4096 (rebuild with
  `--max-chars 16000`) or 2048, keep `batch_size: 1`, and keep
  `gradient_checkpointing: true`.
- **Gated model**: the config uses the public `unsloth/llama-3-8b-Instruct`
  mirror. If you switch to `meta-llama/Meta-Llama-3-8B-Instruct`, accept the
  license on Hugging Face and run `huggingface-cli login` first.
- **Zero training rows**: `train.jsonl` is empty or every row was skipped.
  Rebuild it locally with `node training/scripts/build-dataset.mjs`.
- **Model never stops generating**: run `soup data doctor` and check that the
  assistant turn ends with the EOS token the tokenizer expects.